# Data Cleaning

## Import libraries

In [93]:
import pandas as pd
import numpy as np

## Load Data

In [94]:
df = pd.read_csv("../data/raw/municipal_complaints_raw.csv")

df.head()

,complaint_id,citizen_id,category,sub_category,ward,reported_date,status,people_affected,vulnerable_population,essential_service,public_safety_risk,recurrence_indicator,estimated_effort_hours,severity
0,C000001,P0869,Drainage,Drain Blockage,Peenya,2025-10-01,Closed,230.0,39.0,No,Medium,No,4.0,High
1,C000002,P0993,Sewerage,Sewage Overflow,Hennur,2025-10-01,Resolved,70.0,14.0,Yes,Critical,No,8.0,Medium
2,C000003,P1709,Street Infrastructure,Fallen Tree Branch,Dasarahalli,2025-10-01,Closed,190.0,12.0,No,Medium,No,12.0,Medium
3,C000004,P1458,Revenue Department,Trade Licence Issue,Chickpet,2025-10-01,Resolved,7.0,2.0,No,NaN,No,2.0,Low
4,C000005,P1565,Water Supply,No Water Supply,Vasanthnagar,2025-10-01,Closed,140.0,21.0,Yes,Low,No,12.0,High


## Copy of Data

In [95]:
df_cleaned = df.copy()
df_cleaned.head()

,complaint_id,citizen_id,category,sub_category,ward,reported_date,status,people_affected,vulnerable_population,essential_service,public_safety_risk,recurrence_indicator,estimated_effort_hours,severity
0,C000001,P0869,Drainage,Drain Blockage,Peenya,2025-10-01,Closed,230.0,39.0,No,Medium,No,4.0,High
1,C000002,P0993,Sewerage,Sewage Overflow,Hennur,2025-10-01,Resolved,70.0,14.0,Yes,Critical,No,8.0,Medium
2,C000003,P1709,Street Infrastructure,Fallen Tree Branch,Dasarahalli,2025-10-01,Closed,190.0,12.0,No,Medium,No,12.0,Medium
3,C000004,P1458,Revenue Department,Trade Licence Issue,Chickpet,2025-10-01,Resolved,7.0,2.0,No,NaN,No,2.0,Low
4,C000005,P1565,Water Supply,No Water Supply,Vasanthnagar,2025-10-01,Closed,140.0,21.0,Yes,Low,No,12.0,High


## Clean whitespace in categorical columnn

In [96]:
categorical_columns = df_cleaned.select_dtypes(include="object").columns

for col in categorical_columns:
    df_cleaned[col] = df_cleaned[col].str.strip()
    df_cleaned["category"].unique()

C:\Users\HP\AppData\Local\Temp\ipykernel_1036\387175848.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df_cleaned.select_dtypes(include="object").columns


## Convert date to datetime

In [97]:
df_cleaned["reported_date"] = pd.to_datetime(
    df_cleaned["reported_date"],
    errors="coerce"
)
df_cleaned["reported_date"].dtype

dtype('<M8[us]')

# Handle missing values

In [98]:
df_cleaned.isnull().sum()

complaint_id                 0
citizen_id                   0
category                     0
sub_category                 0
ward                         0
reported_date                0
status                       0
people_affected            161
vulnerable_population      272
essential_service          122
public_safety_risk        1857
recurrence_indicator       179
estimated_effort_hours     219
severity                     0
dtype: int64

### people_affected

In [99]:
df_cleaned["people_affected"] = df_cleaned["people_affected"].fillna(
    df_cleaned["people_affected"].median()
)

### vulnerable_population

In [100]:
df_cleaned["vulnerable_population"] = df_cleaned["vulnerable_population"].fillna(
    df_cleaned["vulnerable_population"].median()
)

### public_safety_risk

In [101]:
df_cleaned["public_safety_risk"] = df_cleaned["public_safety_risk"].fillna(
    df_cleaned["public_safety_risk"].mode()[0]
)

### recurrence_indicator

In [102]:
df_cleaned["recurrence_indicator"] = df_cleaned["recurrence_indicator"].fillna(
    df_cleaned["recurrence_indicator"].mode()[0]
)
df_cleaned.isnull().sum()

complaint_id                0
citizen_id                  0
category                    0
sub_category                0
ward                        0
reported_date               0
status                      0
people_affected             0
vulnerable_population       0
essential_service         122
public_safety_risk          0
recurrence_indicator        0
estimated_effort_hours    219
severity                    0
dtype: int64

In [103]:
df_cleaned.isnull().sum()

complaint_id                0
citizen_id                  0
category                    0
sub_category                0
ward                        0
reported_date               0
status                      0
people_affected             0
vulnerable_population       0
essential_service         122
public_safety_risk          0
recurrence_indicator        0
estimated_effort_hours    219
severity                    0
dtype: int64

# Duplicate complaint records

In [104]:
df_cleaned.duplicated().sum()

np.int64(0)

In [105]:
duplicate_columns = [
    "citizen_id",
    "category",
    "sub_category",
    "ward",
    "reported_date"
]

In [106]:
before = len(df_cleaned)

df_cleaned = df_cleaned.drop_duplicates(
    subset=duplicate_columns,
    keep="first"
)

after = len(df_cleaned)

print("Records removed:", before - after)
print("Records remaining:", after)

Records removed: 264
Records remaining: 8953


In [107]:
df_cleaned.duplicated(subset=duplicate_columns).sum()

np.int64(0)

# Check numerical values for invalid data

In [108]:
df_cleaned[
    ["people_affected", "vulnerable_population"]
].describe()

,people_affected,vulnerable_population
count,8953.000000,8953.000000
mean,251.650955,25.217469
std,1000.113383,75.591028
min,0.000000,0.000000
25%,40.000000,2.000000
50%,95.000000,7.000000
75%,209.000000,21.000000
max,28800.000000,4017.000000


In [109]:
(df_cleaned["people_affected"] < 0).sum()

np.int64(0)

In [110]:
(df_cleaned["vulnerable_population"] < 0).sum()

np.int64(0)

In [111]:
(
    df_cleaned["vulnerable_population"]
    > df_cleaned["people_affected"]
).sum()

np.int64(30)

In [112]:
invalid_population = df_cleaned[
    df_cleaned["vulnerable_population"] > df_cleaned["people_affected"]
]

invalid_population[
    ["complaint_id", "people_affected", "vulnerable_population"]
]

,complaint_id,people_affected,vulnerable_population
98,C000099,0.0,7.0
1562,C001563,50.0,60.0
1800,C001801,220.0,234.0
1842,C001843,95.0,130.0
2575,C002576,0.0,9.0
3555,C003556,6.0,7.0
4004,C004005,95.0,100.0
4437,C004438,0.0,14.0
4572,C004573,0.0,38.0
4739,C004740,35.0,45.0


In [113]:
df_cleaned["vulnerable_population"] = np.minimum(
    df_cleaned["vulnerable_population"],
    df_cleaned["people_affected"]
)

In [114]:
(
    df_cleaned["vulnerable_population"]
    > df_cleaned["people_affected"]
).sum()

np.int64(0)

In [115]:
for col in ["category", "sub_category", "ward", "status",
            "public_safety_risk", "recurrence_indicator", "severity"]:
    print(f"\n{col}")
    print(df_cleaned[col].unique())


category
<ArrowStringArray>
[              'Drainage',               'Sewerage',  'Street Infrastructure',
     'Revenue Department',           'Water Supply',       'Road Maintenance',
    'Road Infrastructure',                 'Health', 'Solid Waste Management',
      'Public Sanitation',             'Electrical',                 'Others']
Length: 12, dtype: str

sub_category
<ArrowStringArray>
[                    'Drain Blockage',                    'Sewage Overflow',
                 'Fallen Tree Branch',                'Trade Licence Issue',
                    'No Water Supply',                      'Water Leakage',
                            'Pothole',          'Road Cutting Not Restored',
     'Public Health Sanitation Issue',                'Sewer Line Blockage',
                    'Footpath Damage',              'Garbage Not Collected',
                   'Manhole Overflow',        'Garbage Vehicle Not Arrived',
              'Missing Manhole Cover',                 'Low 

## Negative Values in Numerical Columns 

In [116]:
for col in ["people_affected", "vulnerable_population"]:
    print(col, (df_cleaned[col] < 0).sum())

people_affected 0
vulnerable_population 0


# Date validity

In [117]:
df_cleaned["reported_date"].isnull().sum()

np.int64(0)

In [118]:
df_cleaned["reported_date"].min(), df_cleaned["reported_date"].max()

(Timestamp('2025-10-01 00:00:00'), Timestamp('2026-09-30 00:00:00'))

# Cleaned dataset

In [119]:
df_cleaned.info()
df_cleaned.shape
df_cleaned.isnull().sum()

<class 'pandas.DataFrame'>
Index: 8953 entries, 0 to 9216
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   complaint_id            8953 non-null   str           
 1   citizen_id              8953 non-null   str           
 2   category                8953 non-null   str           
 3   sub_category            8953 non-null   str           
 4   ward                    8953 non-null   str           
 5   reported_date           8953 non-null   datetime64[us]
 6   status                  8953 non-null   str           
 7   people_affected         8953 non-null   float64       
 8   vulnerable_population   8953 non-null   float64       
 9   essential_service       8834 non-null   str           
 10  public_safety_risk      8953 non-null   str           
 11  recurrence_indicator    8953 non-null   str           
 12  estimated_effort_hours  8737 non-null   float64       
 13  seve

complaint_id                0
citizen_id                  0
category                    0
sub_category                0
ward                        0
reported_date               0
status                      0
people_affected             0
vulnerable_population       0
essential_service         119
public_safety_risk          0
recurrence_indicator        0
estimated_effort_hours    216
severity                    0
dtype: int64

## Save Cleaned Data

In [120]:
df_cleaned.to_csv(
    "../data/processed/cleaned_complaints.csv",
    index=False
)